In this notebook, the main comparison of the search algorithm is run using a PC-KNN, a learned metric, and the energy of the logits over several budgets. We only consider better performing algorithms from the previos test. We still run a single hyperparamter search per dataset, budget, algorithm combination and evalute over several runs.

Dataset can be changed to the values 'bigger_mnist', 'bigger_emnist', 'tu_berlin', and 'modelnet10'.



In [ ]:
import numpy as np
import torch

from its.search import InverseTransformationSearch
from src.utils.sampling import BatchNegativeSampler

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import os

path_found = False
current_path = os.getcwd()
while not path_found:
    if os.path.exists(os.path.join(current_path, "experiment_files")):
        path_found = True
        break
    current_path = os.path.dirname(current_path)

experiment_files_path_data = os.path.join(current_path, "experiment_files", "data")
dataset = "bigger_emnist"

default_architecutre_mapping = {
    "mnist": "resnet_small",
    "bigger_mnist": "resnet_small",
    "emnist": "extended_resnet_small",
    "bigger_emnist": "bigger_extended_resnet_small",
    "tu_berlin": "bi_lstm",
    "modelnet10": "pointnetplus",
}

architecture = default_architecutre_mapping[dataset]
budget = None

In [ ]:
from data.get_dataset import get_dataset_info, get_dataset

dataset_info = get_dataset_info(dataset)

dataset_dict = get_dataset(dataset_info, path=experiment_files_path_data, batch_size=dataset_info.batch_size)
transform_name = dataset_info.transform_seq_name

In [ ]:
dataset_dict.keys()
dataset_train = dataset_dict['train_dataset']
dataset_val = dataset_dict['val_dataset']
dataset_test = dataset_dict['test_dataset']
train_loader = dataset_dict['train_loader']
val_loader = dataset_dict['val_loader']
test_loader = dataset_dict['test_loader']
n_classes = dataset_info.num_classes
train_loader_transformed = dataset_dict['train_loader_transformed']
val_loader_transformed = dataset_dict['val_loader_transformed']
test_loader_transformed = dataset_dict['test_loader_transformed']
train_loader_no_shuffle = dataset_dict['train_loader_no_shuffle']

In [ ]:
from src.utils.eval.vis import vis_dataset

vis_dataset(train_loader, val_loader, test_loader_transformed)

In [ ]:
from model.train import train_and_get_model, train_or_load_energy_model
from model.get_model import get_network

model_dir_path = os.path.join(current_path, "experiment_files", "models")
embedding_cache_path = os.path.join(current_path, "experiment_files", "embedding_cache")
# Add results dir and helper for save paths
results_dir_path = os.path.join(current_path, "experiment_files", "results", dataset, architecture,
                                "comparision_over_budget")
os.makedirs(results_dir_path, exist_ok=True)


def savepath(label: str) -> str:
    safe = "".join(c if c.isalnum() or c in "-_." else "_" for c in label)
    return os.path.join(results_dir_path, transform_name, f"{safe}.json")

In [ ]:
model_dir_path

In [ ]:
model = get_network(dataset_info, architecture, num_classes=n_classes).to(device)
modelname = f"{dataset}_{architecture}"
cache_name_train = f"{dataset}_{architecture}_embedding_cache_train"

train_and_get_model(model, model_dir_path, modelname, train_loader, val_loader, trainer_kwargs={
    "accelerator": "auto",
    "max_epochs": dataset_info.epochs,
    "precision": "16-mixed",
}, load_if_exists=True)



In [ ]:
model.eval().to(device)

In [ ]:
#res = evaluate_base_model(model, test_loader_transformed, device)
#print(res)

In [ ]:
#res = evaluate_base_model(model, test_loader, device)
#print(res)

In [ ]:
class TensorGeometricModelUnwrapper(torch.nn.Module):
    """
    Wrapper for a torch_geometric model that receives a tuple of (pos, y) as input and creates
    a Data object from it that is passed to the model.
    """

    def __init__(self):
        super(TensorGeometricModelUnwrapper, self).__init__()

    def forward(self, data):
        # pos and y are batched tensors from a DataLoader
        # need to reconstruct the original Data objects for torch_geometric models

        pos = data.pos
        batch = data.batch
        #split pos into individual tensors based on batch
        pos_list = torch.split(pos, torch.bincount(batch).tolist())
        return torch.stack(pos_list)

In [ ]:
#chek if data is iamge data
is_image_data = len(dataset_info.input_size) == 3 and dataset_info.input_size[0] in [1, 3]

In [ ]:

from data.transformation import get_transformation_sequence_images

transform_seq = get_transformation_sequence_images(
    name=dataset_info.transform_seq_name,
    resample_method=dataset_info.resample_method,
    init_method="sobol"
).to(device)

In [ ]:
from model.get_model import get_network_layer

layer, layer_io = get_network_layer(dataset_info, architecture, 0, num_classes=None, num_rotations=8)

In [ ]:
from confidence.direct.logit_based import EnergyConfidence
from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence

logit_energy = SinglePassConfidence(model, EnergyConfidence(), index=None)
problem_energy_logits = TransformationProblem(logit_energy, transform_seq, consolidate_method="consolidate_simple")
#test ot
from search.random_search import RSLR

random_search = RSLR(initial_samples=120, local_max_steps=0)

from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search


In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_energy_logits,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("energy_confidence_transformed"), show_progress=True,
    repeats=1)

In [ ]:
model.to(device).eval()

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:


def dec_strat(x, idd, y_true):
    out = model(x)
    eq = out.argmax(dim=-1) == y_true
    #convert to tensor where y>=0 if correct, y<0 if incorrect
    y = torch.where(eq, y_true, -1)
    return y


from src.utils.augments import small_affine_augment_2d
from src.utils.sampling_strategy import TransformLatentSamplingStrategy
import importlib
import src.utils.sampling

importlib.reload(src.utils.sampling)
from src.utils.sampling import BatchNegativeSampler

energy_model2 = get_network(dataset_info, architecture, num_classes=1).to(device)

if is_image_data:
    transform_true_function = small_affine_augment_2d
    affine_augment = src.utils.augments.build_default_augmentations()
else:
    transform_true_function = None
    affine_augment = None

negative_sampling_module = BatchNegativeSampler(
    TransformLatentSamplingStrategy(
        transform_sequence=transform_seq, ), transform_true_function
    =transform_true_function, augment_function=affine_augment,
    decision_strategy=dec_strat,
)

energy_conf2 = train_or_load_energy_model(
    energy_model2, model_dir_path, f"{modelname}_energy2", train_loader,
    val_loader, trainer_kwargs={
        "accelerator": "auto",
        "max_epochs": dataset_info.epochs // 2,
        "precision": "16-mixed" if dataset_info.name not in ["modelnet10"] else "32",
    }, negative_sampling_module=negative_sampling_module, load_if_exists=True)


In [ ]:
model.to(device).eval()

In [ ]:

from model.basic_networks import make_deterministic

# load model should have already called it but twice doesnt hurt
make_deterministic(model)
make_deterministic(energy_model2)

In [ ]:
energy_conf2.to(device).eval()

problem_energy2 = TransformationProblem(energy_conf2, transform_seq, consolidate_method="consolidate_simple")



In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_energy2,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("learned_energy_confidence_transformed"), show_progress=True,
    repeats=1)

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:

from embedding_cache import LayerEmbeddingCache

cache_name_train = f"{dataset}_{architecture}_{transform_name}_embedding_cache_train"

cache_train = LayerEmbeddingCache(model, train_loader_no_shuffle,
                                  cache_dir=os.path.join(embedding_cache_path, cache_name_train))

dual_output_model = cache_train.make_wrapper(layer, capture_modes=layer_io, concat=False, flatten=True)
embeddings_t, final_t, classes_t = cache_train.__call__(layer, capture_modes=layer_io, flatten=True)

from src.utils.transformation_problem import TransformationProblem
from confidence.model.single_pass import SinglePassConfidence
from confidence.direct.logit_based import EnergyConfidence
from confidence.control.split import PredictedSplitConfidence
from confidence.unsupervised.classic.nn_pytorch import PerClassKNNConfidence

nn_pytorch_pretrained = PerClassKNNConfidence(metric="cosine", input_transform=None, computation_mode="masked", k=1)
nn_pytorch_pretrained.fit(embeddings_t, classes_t)
nn_pytorch_pretrained.to(device)

conf_split_pretrained = PredictedSplitConfidence(nn_pytorch_pretrained, EnergyConfidence(), mult=False, b=0.0)
conf_mod_nn_pytorch_pretrained = SinglePassConfidence(dual_output_model, conf_split_pretrained, index=1)
problem_nn_pytorch_pretrained = TransformationProblem(conf_mod_nn_pytorch_pretrained, transform_seq,
                                                      consolidate_method="consolidate_simple")
model.eval().to(device)

In [ ]:
x = next(iter(train_loader_no_shuffle))

res1 = dual_output_model(x[0].to(device))[0].cpu().detach().numpy()

res2 = embeddings_t[:x[0].shape[0]].cpu().detach().numpy()
if not np.allclose(res1, res2, rtol=1e-2, atol=1e-2):
    raise ValueError("Model is not deterministic!")
else:
    print("Model is deterministic.")

del res1
del res2
del x
#There seem to be a missmatch since i updated pytorch and the graphics card wihch is why i decreased tolerances. As i am not recalculating any results this should not matter but one should be carefull with the cache. Increasing tolerance solves this but one should likely recalculate the caches with the new database if any chages are made.
# This means that changing these settings can introduce a quite large differenceeven if all other things are fixed.

In [ ]:
#benchmark model and dual output model

In [ ]:
load_or_run_evaluate_confidence_and_search(
    model, optimizer=random_search, problem=problem_nn_pytorch_pretrained,
    test_loader=test_loader_transformed, max_batch_override=dataset_info.batch_size_search,
    save_path=savepath("knn_per_class_confidence_transformed"), show_progress=True,
    repeats=1, overwrite=False)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.utils.eval.ood_performance import ITSWRAPPER
import importlib
import its.search

importlib.reload(its.search)

its2 = ITSWRAPPER(its.search.InverseTransformationSearch(model, None, None, n_hypotheses=1, n_samples=10, extend=0,
                                                         gaussian_filter_channel_wise=True))

In [ ]:
from search.coordinate_descent import CoordinateDescent

cd = CoordinateDescent()

In [ ]:
gc.collect(

)
torch.cuda.empty_cache()

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
x = next(iter(test_loader_transformed))[0].to(device)

In [ ]:

from src.utils.transformation_problem import TransformationProblem



In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
from src.utils.replacer import replace_rotation_transforms, replace_rotation_transforms_2vec
import os
import optuna
import torch
import gc
import numpy as np
from torch.utils.data import Subset, DataLoader

from hyper_param.search.objective_generators import (
    make_search_objective,
    save_best_trial_params,
    build_search_algorithm,
    _cost_shgo,
)
from hyper_param.search.default_values import load_params, get_default_params, save_params
from src.utils.eval.ood_performance import load_or_run_evaluate_confidence_and_search

model.eval().to(device)
# detach model for efficent gradients
for param in model.parameters():
    param.requires_grad = False

for param in energy_model2.parameters():
    param.requires_grad = False

# Algorithms and budgets
all_algos = ["shgo", "wcd_lattice", "its", "random_search"]

#for bigger mnist and bigger emnist add shgo
#if dataset in ["bigger_mnist","bigger_emnist","coil100"]:
#   all_algos.append("shgo")


# remove pgd pgd restarts and pgd window size for tu berlin
if dataset in ["tu_berlin"]:
    all_algos = ["wcd_lattice", "its", "random_search"]
    grad_weight = 99999999
elif dataset in ["modelnet10"]:
    all_algos = ["shgo_vector", "wcd_lat_ind", "random_search", "its"]
    grad_weight = 2
else:
    grad_weight = 2

# Add complex/quaternion variants
complex_algos = ["shgo_c", "pgd_c", "shgo_c_3"]

budgets = [8, 15, 30, 60, 120, 240]

grad_weight_algos = {"shgo", "shgo_individual", "pgd", "pgd_restart", "pgd_window",
                     "shgo_c", "pgd_c", "pgd_restart_c", "shgo_c_3", "pgd_c_3", "pgd_vector", "shgo_vector"}
default_trials = 30
eval_repeats = 5
if dataset == "modelnet10" or dataset == "tu_berlin" or dataset == "coil100":
    eval_repeats = 10
show_progress = True

# Problem configurations
problems = [problem_energy_logits, problem_nn_pytorch_pretrained, problem_energy2]
problem_names = ["logit_energy", "knn_per_class", "learned_energy"]

# Create complex/quaternion versions of problems
problems_complex = [replace_rotation_transforms(p) for p in problems]
problem_names_complex = problem_names

# New problems for shgo_individual
problems_individual = [ITSWRAPPER._prepare_problem(p) for p in problems]
problem_names_individual = problem_names

problems_rotvec = [replace_rotation_transforms_2vec(p) for p in problems]
problem_names_rotvec = problem_names

for p in problems:
    p.max_batch_size = dataset_info.batch_size_search
for p in problems_individual:
    p.max_batch_size = dataset_info.batch_size_search
for p in problems_complex:
    p.max_batch_size = dataset_info.batch_size_search

# Base results directory
base_results_dir = os.path.join(
    current_path, "experiment_files", "search_results_compare_budget",
    str(dataset), dataset_info.transform_seq_name, str(architecture)
)
os.makedirs(base_results_dir, exist_ok=True)

assert len(problems) == len(problem_names), "Mismatch between problems and problem_names."

# Mapping from complex variants to their base algorithms
algo_variant_mapping = {
    "pgd_c": "pgd",
    "shgo_c": "shgo",
    "pgd_restart_c": "pgd_restart",
    "shgo_c_3": "shgo",
    "pgd_c_3": "pgd",
    "pgd_vector": "pgd",
    "shgo_vector": "shgo",
}

# Create subsampled validation loaders (before budget loop)
val_dataset_original = val_loader_transformed.dataset
n_val = len(val_dataset_original)
rng_seed = 42
rng = np.random.default_rng(rng_seed)
perm = rng.permutation(n_val)

half_n = n_val // 2
quarter_n = n_val // 4

indices_half = perm[:half_n]
indices_quarter = perm[:quarter_n]

# Create half-size validation loader
val_loader_transformed_half = DataLoader(
    Subset(val_dataset_original, indices_half),
    batch_size=max(dataset_info.batch_size // 2, 1),
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=getattr(val_loader_transformed, "persistent_workers", False),
)

# Create quarter-size validation loader
val_loader_transformed_quarter = DataLoader(
    Subset(val_dataset_original, indices_quarter),
    batch_size=max(dataset_info.batch_size // 4, 1),
    shuffle=False,
    num_workers=val_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=getattr(val_loader_transformed, "persistent_workers", False),
)

test_loader_lower_batch_size = DataLoader(
    test_loader_transformed.dataset,
    batch_size=max(dataset_info.batch_size // 2, 1),
    shuffle=False,
    num_workers=test_loader_transformed.num_workers,
    pin_memory=True,
    persistent_workers=getattr(test_loader_transformed, "persistent_workers", False),
)

grad_weight_orig = grad_weight
for budget in budgets:
    print(f"\n=== Budget: {budget} ===")
    for algo in all_algos:
        # Determine validation loader and eval repeats based on budget
        if budget > 200:
            val_dataset_transformed_loader = val_loader_transformed_quarter
            repeats_for_eval = eval_repeats // 2
        elif budget > 100:
            val_dataset_transformed_loader = val_loader_transformed_half
            repeats_for_eval = (eval_repeats * 3) // 4
        else:
            val_dataset_transformed_loader = val_loader_transformed
            repeats_for_eval = eval_repeats
        if "its" in algo:
            repeats_for_eval = 1

        gc.collect()
        torch.cuda.empty_cache()
        print(f"\n--- Algorithm: {algo} ---")

        if algo == "shgo_c_3" or algo == "pgd_c_3":
            grad_weight = 3
        else:
            grad_weight = grad_weight_orig

        # Determine which problems to use
        if algo in complex_algos:
            current_problems = problems_complex
            current_problem_names = problem_names_complex
        elif algo in ["shgo_individual", "wcd_lat_ind"]:
            current_problems = problems_individual
            current_problem_names = problem_names_individual
        elif algo in ["pgd_vector", "shgo_vector"]:
            current_problems = problems_rotvec
            current_problem_names = problem_names_rotvec
        else:
            current_problems = problems
            current_problem_names = problem_names

        for p in current_problems:
            p.max_batch_size = dataset_info.batch_size_search

        # Map algorithm names for construction
        if algo in complex_algos:
            algo_name_for_path = algo_variant_mapping[algo]
        elif algo == "shgo_individual":
            algo_name_for_path = "shgo"
        elif algo == "wcd_lat_ind":
            algo_name_for_path = "wcd_lattice"
        elif algo in ["pgd_vector", "shgo_vector"]:
            algo_name_for_path = algo_variant_mapping[algo]
        else:
            algo_name_for_path = algo

        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        os.makedirs(algo_dir, exist_ok=True)

        param_path = os.path.join(algo_dir, "best.yml")
        print(f"Result directory: {algo_dir}")

        # Load stored params or optimize
        stored_params = load_params(param_path) if os.path.exists(param_path) else None
        if stored_params is None and (algo != "cd" and "random_search" not in algo):
            default_params_kwargs = {}
            if algo in grad_weight_algos:
                default_params_kwargs["grad_weight"] = grad_weight
            default_params = get_default_params(algo_name_for_path, budget, **default_params_kwargs)
            print("Default params (config):", default_params)

            objective_kwargs = {}
            if algo in grad_weight_algos:
                objective_kwargs["grad_weight"] = grad_weight

            objective = make_search_objective(
                algo=algo_name_for_path,
                model=model,
                val_loader=val_dataset_transformed_loader,  # Use budget-dependent validation loader
                problem=current_problems,  # multi-problem objective
                budget=budget,
                device=str(device),
                repeats=1,
                **objective_kwargs,
            )

            pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0, interval_steps=1)
            study = optuna.create_study(direction="maximize", pruner=pruner)
            study.enqueue_trial(default_params)

            study.optimize(objective, n_trials=default_trials, show_progress_bar=False)

            print(f"[{algo}] Best validation value:", study.best_value)
            print("Suggested params:", study.best_trial.params)
            print("Full params:", study.best_trial.user_attrs.get("full_params"))

            save_best_trial_params(study, algo=algo_name_for_path, path=param_path)
            stored_params = load_params(param_path)
            print("Saved best params to:", param_path)
        else:
            print("Found stored best params, skipping optimization.")
            if "cd" == algo:
                print("Note: 'cd' algorithm requires no hyperparameter optimization.")
                stored_params = {}

        # SHGO-only sanity check: force full budget consumption by topping up n_init
        if algo in ["shgo", "shgo_individual", "shgo_c", "shgo_c_3"]:
            gw = stored_params.get("grad_weight", 2)
            cost = _cost_shgo(
                stored_params["shgo_initial_samples"],
                stored_params["shgo_local_runs"],
                stored_params["shgo_local_steps"],
                gw,
            )
            if cost != budget:
                delta = budget - cost
                # Adjust n_init up/down to match budget exactly
                stored_params["shgo_initial_samples"] = max(
                    1,
                    stored_params["shgo_initial_samples"] + delta
                )
                # Recompute and assert
                new_cost = _cost_shgo(
                    stored_params["shgo_initial_samples"],
                    stored_params["shgo_local_runs"],
                    stored_params["shgo_local_steps"],
                    gw,
                )
                print(f"[{algo}] adjusted n_init by {delta} to match budget: {cost} -> {new_cost}")
                assert new_cost == budget, f"SHGO cost mismatch after fix: {new_cost}!={budget}"
                # Persist the fix
                save_params(stored_params, param_path)

        # assert that grad weight matches
        if algo in grad_weight_algos:
            assert stored_params.get("grad_weight", None) == grad_weight, \
                f"Grad weight mismatch for {algo}: {stored_params.get('grad_weight', None)}!={grad_weight}"

        # Rebuild optimizer with best params
        search_obj = build_search_algorithm(
            algo_name_for_path,
            stored_params,
            problem=current_problems[0],  # any problem is fine for optimizer construction
            budget=budget,
            model=model,
        )
        print("Rebuilt search object from saved params.")

        # Evaluate per problem with cached runner
        for prob, method_name in zip(current_problems, current_problem_names):
            eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
            print(f"[{method_name}] evaluating (cached path: {eval_path})")
            metrics = load_or_run_evaluate_confidence_and_search(
                model=model,
                optimizer=search_obj,
                problem=prob,
                test_loader=test_loader_transformed if budget < 200 else test_loader_lower_batch_size,
                save_path=eval_path,  # auto-loads if exists; saves after run otherwise
                max_batch_override=dataset_info.batch_size_search,
                show_progress=show_progress,
                repeats=repeats_for_eval,  # Use budget-dependent repeats
                return_per_run=True,
                overwrite=False, store_val=True, rerun_if_duplicates=True if not "its" in algo else False,
            )
            gc.collect()
            torch.cuda.empty_cache()

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [ ]:
import pandas as pd
import json
import os

# Collect results
results = []

for budget in budgets:
    for algo in all_algos:
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        for method_name in problem_names:
            eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
            if os.path.exists(eval_path):
                with open(eval_path, "r") as f:
                    metrics = json.load(f)
                accuracy = metrics.get("accuracy_mean", None)
                accuracy_se = metrics.get("accuracy_se", None)
                results.append({
                    "Algorithm": algo,
                    "Budget": budget,
                    "Problem": method_name,
                    "Accuracy": accuracy,
                    "Accuracy_SE": accuracy_se,

                })

# Convert to DataFrame
df = pd.DataFrame(results)




In [ ]:
df

In [ ]:
transform_seq.transformations

In [ ]:

import pandas as pd
import json
import os

# Collect results
results = []

for budget in budgets:
    for algo in all_algos:
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        for method_name in problem_names:
            eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
            if os.path.exists(eval_path):
                with open(eval_path, "r") as f:
                    metrics = json.load(f)

                accuracy = metrics.get("accuracy_mean", None)
                accuracy_se = metrics.get("accuracy_se", None)
                accuracy_std = metrics.get("accuracy_std", None)
                number_of_runs = metrics.get("repeats", None)
                results.append({
                    "Algorithm": algo,
                    "Budget": budget,
                    "Problem": method_name,
                    "Accuracy": accuracy,
                    "Accuracy_SE": accuracy_se,
                    "Accuracy_STD": accuracy_std,
                    "Number_of_Runs": number_of_runs,

                })

# Convert to DataFrame
df = pd.DataFrame(results)

df
transform_seq.transformations
ALGO_RENAME = {
    "cd": "CD-S",
    "cd_multi_cyclus": "CD",
    "shgo": "RS-LR",
    "parallel_sa": "PSA",
    "pso": "PSO",
    "pgd": "GD",
    "pgd_restart": "GD-R",
    "its": "ITS",
    "random_search": "R. Search",
    "wcd": "WCD",
    "wcd_lattice": "WCD-L",
    "shgo_c": "RS-LR-C",
    "pgd_c": "GD-C",
    "wcd_lat_ind": "WCD-Lat-EUL",
    "pgd_vector": "GD-Vec",
    "shgo_vector": "RS-LR-Vec",
}

PROBLEM_RENAME = {
    "knn_per_class": "PC-kNN",
    "logit_energy": "Logit Energy",
    "learned_energy": "Learned Energy",
}

#do some renaming

df_renamed = df.copy()
#rename cd to cd_signle_cycle
#rename cd_multi_cyclus to cd
#rename shgo to RS-LR
df_renamed["Algorithm"] = df["Algorithm"].replace(ALGO_RENAME)

#reanme problem name from knn_per_class to PC-kNN
df_renamed["Problem"] = df["Problem"].replace(PROBLEM_RENAME)
problem_names_renamed = ["Logit Energy", "PC-kNN", "Learned Energy"]

fullname_dict = {
    "RS-LR-Vec": "Random Search with Local Refinement (rotation vector representation)",
    "CD": "Coordinate Descent",
    "RS-LR": "Random Search with Local Refinement",
    "ITS": "Inverse Transformation Search",
    "R. Search": "Random Search",
    "PSA": "Parallel Simulated Annealing",
    "PSO": "Particle Swarm Optimization",
    "GD": "Multistart Gradient Descent",
    "WCD-L-EUL": "Weighted Coordinate Descent with Lattice sampling (euler angles)",
    "WCD-L": "Weighted Coordinate Descent with Lattice sampling",
    "WCD": "Weighted Coordinate Descent",
    "GD-R": "Multistart Gradient Descent with Restarts",
    "RS-LR-C": "Random Search with Local Refinement (complex/quaternion rotations)",
    "GD-C": "Multistart Gradient Descent (complex/quaternion rotations)",
    "GD-Vec": "Multistart Gradient Descent (rotation vector representation)",
    "CD-S": "Coordinate Descent (single cycle)",
}

import matplotlib.pyplot as plt

short_names = list(fullname_dict.keys())
n = len(short_names)

# Get all 20 colors from tab20
tab20_colors = [plt.get_cmap("tab20")(i)[:3] for i in range(20)]

# Separate even and odd indices
even_colors = tab20_colors[::2]  # 0, 2, 4, ..., 18
odd_colors = tab20_colors[1::2]  # 1, 3, 5, ..., 19

# Combine: even first, then odd
ordered_colors = even_colors + odd_colors

# Repeat pattern if more names than 20
if n > 20:
    repeats = (n // 20) + 1
    ordered_colors = (ordered_colors * repeats)[:n]
else:
    ordered_colors = ordered_colors[:n]

# Assign to names
algorithm_colors = {name: ordered_colors[i] for i, name in enumerate(short_names)}

figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "comparision_search_datasets", dataset,
                           transform_name)
if not os.path.exists(figure_path):
    os.makedirs(figure_path)

from matplotlib.patches import Patch
from src.utils.eval.vis import plt_setup_latex

W = plt_setup_latex()

# Sort algorithm keys alphabetically
sorted_shorts = sorted(fullname_dict.keys())

handles = [
    Patch(color=algorithm_colors[short], label=f"{short} — {fullname_dict[short]}")
    for short in sorted_shorts
]

# Plot legend in a separate figure
fig, ax = plt.subplots(figsize=(W, W * 0.4))
ax.legend(
    handles=handles,
    loc='center',
    frameon=False,
    fontsize=9
)
ax.axis('off')
plt.savefig(os.path.join(figure_path, "comparision_search_algorithms_legend.pdf"), bbox_inches='tight')
plt.savefig(os.path.join(figure_path, "comparision_search_algorithms_legend.pgf"), bbox_inches='tight')
plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

figure_path = os.path.join(current_path, "experiment_files", "export", "fig", "comparision_search_budget", dataset,
                           transform_name)
if not os.path.exists(figure_path):
    os.makedirs(figure_path)

In [ ]:
#load its best hyperparameter per budget and print the number of hypothesis. Similarly for wcd_lattice print the number of rounds
import os
import yaml

for budget in budgets:
    for algo in ["shgo_c"]:
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        param_path = os.path.join(algo_dir, "best.yml")
        if os.path.exists(param_path):
            with open(param_path, "r") as f:
                params = yaml.safe_load(f)
            if algo == "shgo_c":
                n_rounds = params.get("n_rounds", None)
                print(f"[Budget {budget}] WCD Lattice n_rounds: {n_rounds}")
                print(params)

In [ ]:
def analyze_run_results_with_budget(results_list):
    """
    Computes summary statistics per algorithm and budget for each problem.
    Uses global min-max normalization across all budgets, runs, and algorithms.
    Frobenius distance is computed against the global best run per sample.
    """
    import numpy as np
    from collections import defaultdict

    # Apply renaming first, then group
    for entry in results_list:
        algo = ALGO_RENAME.get(entry["Algorithm"], entry["Algorithm"])
        problem_name = PROBLEM_RENAME.get(entry["Problem"], entry["Problem"])
        entry["Algorithm"] = algo
        entry["Problem"] = problem_name

    results_by_problem = defaultdict(list)
    for entry in results_list:
        results_by_problem[entry["Problem"]].append(entry)

    problem_summaries = {}

    for problem, entries in results_by_problem.items():
        # First pass: collect all errors globally to find min/max and best run per sample
        all_errors_global = []
        all_mats_global = []
        metadata_global = []
        all_true_labels, all_pred_labels = [], []

        for entry in entries:
            algo = entry["Algorithm"]
            problem_name = entry["Problem"]
            budget = entry["Budget"]

            res = entry["metrics"]
            runs = res.get("per_run", [res])

            for run_idx, run in enumerate(runs):
                if "per_sample_errors" not in run or "per_sample_matrices" not in run:
                    continue

                errs = np.array(run["per_sample_errors"], dtype=float)
                mats = run["per_sample_matrices"]

                all_errors_global.append(errs)
                all_mats_global.append(mats)
                metadata_global.append({
                    'algo': algo,
                    'budget': budget,
                    'run_idx': run_idx
                })

                if "per_sample_true_labels" in run:
                    all_true_labels.append(run["per_sample_true_labels"])
                if "per_sample_pred_labels" in run:
                    all_pred_labels.append(run["per_sample_pred_labels"])

        if not all_errors_global:
            continue

        array_labels = np.array(all_true_labels)
        assert np.all(array_labels == array_labels[0]), "Rows are not equal"

        all_errors_global = np.stack(all_errors_global)
        eps = 1e-12
        num_samples = all_errors_global.shape[1]

        # Global min/max across all budgets and algorithms
        min_errors = np.nanmin(all_errors_global, axis=0)
        max_errors = np.nanmax(all_errors_global, axis=0)
        denom = max_errors - min_errors + eps

        # Global best run per sample
        best_run_idx_per_sample = np.nanargmin(
            np.where(np.isnan(all_errors_global), np.inf, all_errors_global), axis=0
        )
        best_mats = [all_mats_global[idx][j] for j, idx in enumerate(best_run_idx_per_sample)]

        # Best predicted labels
        best_pred_labels = None
        pred_labels_available = len(all_pred_labels) == len(all_errors_global)
        if pred_labels_available:
            best_pred_labels = [
                all_pred_labels[idx][j] for j, idx in enumerate(best_run_idx_per_sample)
            ]

        # Vectorized relative error computation
        relative_errors_global = (all_errors_global - min_errors[None, :]) / denom[None, :]

        # Pre-compute Frobenius distances

        best_mats_array = np.array(best_mats, dtype=float)
        all_mats_stacked = []
        for run_mats in all_mats_global:
            run_mats_array = np.array(run_mats[:num_samples], dtype=float)
            all_mats_stacked.append(run_mats_array)
        all_mats_stacked = np.array(all_mats_stacked)

        diff = all_mats_stacked - best_mats_array[None, :, ...]
        shape = diff.shape
        diff_flat = diff.reshape(shape[0], shape[1], -1)
        best_flat = best_mats_array.reshape(shape[1], -1)

        diff_norms = np.linalg.norm(diff_flat, axis=2)
        best_norms = np.linalg.norm(best_flat, axis=1)

        frobenius_distances_global = diff_norms / (best_norms[None, :] + eps)
        valid_mask = ~np.isnan(all_errors_global) & np.isfinite(frobenius_distances_global)
        frobenius_distances_global = np.where(valid_mask, frobenius_distances_global, np.nan)

        # Second pass: organize by budget and algorithm
        summary_by_budget = defaultdict(dict)

        for entry in entries:
            algo = entry["Algorithm"]
            budget = entry["Budget"]
            res = entry["metrics"]
            runs = res.get("per_run", [res])

            # Find indices for this algo+budget combination
            indices = [i for i, meta in enumerate(metadata_global)
                       if meta['algo'] == algo and meta['budget'] == budget]

            if not indices:
                continue

            algo_rel_errs = relative_errors_global[indices]
            algo_frobs = frobenius_distances_global[indices]

            run_avg_rel_errs = np.nanmean(algo_rel_errs, axis=1)
            run_avg_rel_errs = run_avg_rel_errs[~np.isnan(run_avg_rel_errs)]

            run_avg_frobs = np.nanmean(algo_frobs, axis=1)
            run_avg_frobs = run_avg_frobs[~np.isnan(run_avg_frobs)]

            num_runs = len(run_avg_rel_errs)

            summary_by_budget[budget][algo] = {
                "mean_relative_error": float(np.mean(run_avg_rel_errs)) if len(run_avg_rel_errs) > 0 else None,
                "std_relative_error": float(np.std(run_avg_rel_errs, ddof=1)) if num_runs > 1 else 0,
                "mean_frobenius": float(np.mean(run_avg_frobs)) if len(run_avg_frobs) > 0 else None,
                "std_frobenius": float(np.std(run_avg_frobs, ddof=1)) if num_runs > 1 else 0,
                "se_relative_error": float(np.std(run_avg_rel_errs, ddof=1) / np.sqrt(num_runs)) if num_runs > 1 else 0,
                "se_frobenius": float(np.std(run_avg_frobs, ddof=1) / np.sqrt(num_runs)) if len(
                    run_avg_frobs) > 1 else 0,
                "num_runs": num_runs
            }

        problem_summaries[problem] = {
            "num_datapoints": num_samples,
            "budgets": dict(summary_by_budget),
            "true_labels": all_true_labels[0],
            "best_predicted_labels": best_pred_labels,
        }

    return problem_summaries


# Collect results with budget information
results_list = []
for budget in budgets:
    for algo in all_algos:
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        for method_name in problem_names:
            eval_path = os.path.join(algo_dir, f"eval_{method_name}.json")
            if os.path.exists(eval_path):
                with open(eval_path, "r") as f:
                    metrics = json.load(f)
                results_list.append({
                    "Algorithm": algo,
                    "Problem": method_name,
                    "Budget": budget,
                    "metrics": metrics
                })

# Analyze with budget information
analysis = analyze_run_results_with_budget(results_list)


def fmt(x):
    return f"{x:.6f}" if isinstance(x, (float, int)) else "nan"


# Print summary
for problem, pdata in analysis.items():
    print(f"\n=== Problem: {problem} ===")
    print(f"Analyzed {pdata['num_datapoints']} datapoints")
    for budget in sorted(pdata["budgets"].keys()):
        print(f"\n  Budget: {budget}")
        for algo, stats in pdata["budgets"][budget].items():
            print(
                f"    {algo}: "
                f"RelErr={fmt(stats['mean_relative_error'])}±{fmt(stats['se_relative_error'])}, "
                f"Frob={fmt(stats['mean_frobenius'])}±{fmt(stats['se_frobenius'])}"
            )


In [ ]:
df_renamed

rename_mapping = {"WCD-Lat-EUL": "WCD-L", "RS-LR-Vec": "RS-LR"}

df_renamed["Algorithm"] = df_renamed["Algorithm"].replace(rename_mapping)


In [ ]:
from src.utils.eval.vis import plt_setup_paper
import os

W = plt_setup_paper()
figure_path2 = os.path.join(current_path, "experiment_files", "export", "fig2", "comparison_search_datasets", dataset,
                            transform_name)


def plot_accuracy_vs_budget_side_by_side(
        df,
        problem_names,
        algo_column="Algorithm",
        problem_column="Problem",
        x_column="Budget",
        y_column="Accuracy",
        se_column="Accuracy_SE",
        algorithm_colors=None,
        fig_width=10,
        fig_height=None,
        figure_path=None,
        save_name="accuracy_vs_budget_side_by_side"
):
    """
    Plots Accuracy vs Budget curves for each problem side-by-side with error bars,
    and optionally saves the figure as both PDF and PGF.

    Parameters:
        df : pd.DataFrame
            DataFrame containing results.
        problem_names : list
            List of problem names to plot (order matters).
        algo_column : str
            Column name for algorithm labels.
        problem_column : str
            Column name for problem names.
        x_column : str
            Column for x-axis values (e.g. "Budget").
        y_column : str
            Column for y-axis values (e.g. "Accuracy").
        se_column : str
            Column for standard errors.
        algorithm_colors : dict
            Optional mapping {algorithm: color}.
        fig_width, fig_height : float
            Dimensions of the overall figure.
        figure_path : str
            Path to directory for saving the figure.
        save_name : str
            Base name for output file (without extension).
    """
    #remove its with budget 8
    df = df[~((df[algo_column] == "ITS") & (df[x_column] == 8))]

    sns.set(style="whitegrid", context="paper")
    W = plt_setup_paper()
    n_problems = len(problem_names)
    if fig_height is None:
        fig_height = fig_width * 0.4

    fig, axes = plt.subplots(
        1, n_problems,
        figsize=(fig_width, fig_height),
        sharey=True
    )

    if n_problems == 1:
        axes = [axes]

    for ax, problem in zip(axes, problem_names):
        problem_df = df[df[problem_column] == problem]

        for algorithm in problem_df[algo_column].unique():
            alg_df = problem_df[problem_df[algo_column] == algorithm]

            color = None
            if algorithm_colors and algorithm in algorithm_colors:
                color = algorithm_colors[algorithm]

            ax.errorbar(
                alg_df[x_column],
                alg_df[y_column],
                yerr=alg_df[se_column] * 1.96,
                marker="o",
                label=algorithm,
                capsize=3,
                linewidth=1.5,
                markersize=1.6,
                color=color,
                alpha=0.7
            )

        ax.set_title(problem, fontsize=10)
        ax.set_xlabel("Budget", fontsize=8)
        if ax == axes[0]:
            ax.set_ylabel("Accuracy", fontsize=8)
        else:
            ax.set_ylabel("")
        ax.tick_params(axis='both', labelsize=8)
        ax.grid(axis="y", linestyle=":", alpha=0.8, zorder=-1)
        sns.despine(ax=ax)

    # Shared legend at top
    handles, labels = axes[0].get_legend_handles_labels()

    left, top = 1.0, 0.27
    if dataset == "modelnet10":
        left = 1.0
        top = 0.25
    if dataset == "tu_berlin":
        left = 1.0
        top = 0.27
    if dataset == "coil100":
        left = 1.0
        top = 0.54

    fig.legend(
        handles, labels, title="",
        loc="lower right", ncol=len(labels) // 2,
        bbox_to_anchor=(left, top),
        fontsize=8, title_fontsize=8,
        columnspacing=0.3, framealpha=0.5, labelspacing=0.3
    )

    #plt.tight_layout(pad=0.6, w_pad=0.3)
    plt.tight_layout()

    # Save as PDF and PGF if path provided
    if figure_path is not None:
        os.makedirs(figure_path, exist_ok=True)
        pdf_path = os.path.join(figure_path, f"{save_name}.pdf")
        plt.savefig(pdf_path, bbox_inches='tight')
        print(f"Saved figure to:\n  {pdf_path}")

    plt.show()


W = plt_setup_paper()

plot_accuracy_vs_budget_side_by_side(
    df=df_renamed,
    problem_names=problem_names_renamed,
    algorithm_colors=algorithm_colors,
    fig_width=W,
    figure_path=figure_path2
)


In [ ]:
from src.utils.eval.vis import plt_setup_paper
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

W = plt_setup_latex(W=5.53)
figure_path3 = os.path.join(current_path, "experiment_files", "export", "fig3", "comparison_search_datasets", dataset,
                            transform_name)


def plot_accuracy_vs_budget_side_by_side(
        df,
        problem_names,
        algo_column="Algorithm",
        problem_column="Problem",
        x_column="Budget",
        y_column="Accuracy",
        se_column="Accuracy_SE",
        algorithm_colors=None,
        fig_width=10,
        fig_height=None,
        figure_path=None,
        save_name="accuracy_vs_budget_side_by_side"
):
    """
    Plots Accuracy vs Budget curves for each problem side-by-side with error bars,
    and optionally saves the figure as both PDF and PGF.

    Parameters:
        df : pd.DataFrame
            DataFrame containing results.
        problem_names : list
            List of problem names to plot (order matters).
        algo_column : str
            Column name for algorithm labels.
        problem_column : str
            Column name for problem names.
        x_column : str
            Column for x-axis values (e.g. "Budget").
        y_column : str
            Column for y-axis values (e.g. "Accuracy").
        se_column : str
            Column for standard errors.
        algorithm_colors : dict
            Optional mapping {algorithm: color}.
        fig_width, fig_height : float
            Dimensions of the overall figure.
        figure_path : str
            Path to directory for saving the figure.
        save_name : str
            Base name for output file (without extension).
    """
    #remove its with budget 8
    df = df[~((df[algo_column] == "ITS") & (df[x_column] == 8))]

    sns.set(style="whitegrid", context="paper")
    W = plt_setup_latex(W=5.53)
    n_problems = len(problem_names)
    if fig_height is None:
        fig_height = fig_width * 0.36

    fig, axes = plt.subplots(
        1, n_problems,
        figsize=(fig_width, fig_height),
        sharey=True
    )

    if n_problems == 1:
        axes = [axes]

    for ax, problem in zip(axes, problem_names):
        problem_df = df[df[problem_column] == problem]

        for algorithm in problem_df[algo_column].unique():
            alg_df = problem_df[problem_df[algo_column] == algorithm]

            color = None
            if algorithm_colors and algorithm in algorithm_colors:
                color = algorithm_colors[algorithm]

            ax.errorbar(
                alg_df[x_column],
                alg_df[y_column],
                yerr=alg_df[se_column] * 1.96,
                marker="o",
                label=algorithm,
                capsize=3,
                linewidth=1.5,
                markersize=1.6,
                color=color,
                alpha=0.7
            )

        ax.set_title(problem, fontsize=10)
        ax.set_xlabel("Budget", fontsize=8)
        if ax == axes[0]:
            ax.set_ylabel("Accuracy", fontsize=10)
        else:
            ax.set_ylabel("")
        ax.tick_params(axis='both', labelsize=8)
        ax.grid(axis="y", linestyle=":", alpha=0.8, zorder=-1)
        sns.despine(ax=ax)

    # Shared legend at top
    handles, labels = axes[0].get_legend_handles_labels()

    left, top = 0.873, 0.21
    if dataset == "modelnet10":
        left = 1.0
        top = 0.25
    if dataset == "tu_berlin":
        left = 1.0
        top = 0.27
    if dataset == "coil100":
        left = 1.0
        top = 0.54

    fig.legend(
        handles, labels, title="",
        loc="lower right", ncol=len(labels) // 2,
        bbox_to_anchor=(left, top),
        fontsize=8, title_fontsize=8,
        columnspacing=0.3, framealpha=0.5, labelspacing=0.3
    )

    #plt.tight_layout(pad=0.6, w_pad=0.3)
    plt.tight_layout()

    # Save as PDF and PGF if path provided
    if figure_path is not None:
        os.makedirs(figure_path, exist_ok=True)
        pdf_path = os.path.join(figure_path, f"{save_name}.pdf")
        plt.savefig(pdf_path, bbox_inches='tight')
        print(f"Saved figure to:\n  {pdf_path}")

    plt.show()


W = plt_setup_latex(W=5.5325)
plot_accuracy_vs_budget_side_by_side(
    df=df_renamed,
    problem_names=problem_names_renamed,
    algorithm_colors=algorithm_colors,
    fig_width=W,
    figure_path=figure_path3
)


In [ ]:
from collections import defaultdict

W = plt_setup_latex(W=5.53)


def plot_error_metrics_vs_budget(
        analysis,
        problem_names,
        metric="mean_relative_error",
        se_key="se_relative_error",
        algorithm_colors=None,
        fig_width=10,
        fig_height=None,
        figure_path=None,
        save_name="error_vs_budget"
):
    """
    Plots error metrics vs budget for each problem side-by-side.

    Parameters:
        analysis : dict
            Output from analyze_run_results_with_budget
        problem_names : list
            List of problem names to plot
        metric : str
            Metric to plot (e.g., "mean_relative_error" or "mean_frobenius")
        se_key : str
            Standard error key
        algorithm_colors : dict
            Color mapping for algorithms
        fig_width, fig_height : float
            Figure dimensions
        figure_path : str
            Directory for saving
        save_name : str
            Base filename
    """
    import matplotlib.pyplot as plt
    import seaborn as sns

    sns.set(style="whitegrid", context="paper")
    W = plt_setup_latex(W=5.53)

    n_problems = len(problem_names)
    if fig_height is None:
        fig_height = fig_width * 0.36

    fig, axes = plt.subplots(1, n_problems, figsize=(fig_width, fig_height), sharey=True, dpi=300)

    if n_problems == 1:
        axes = [axes]

    for ax, problem in zip(axes, problem_names):
        pdata = analysis[problem]
        budgets_data = pdata["budgets"]

        # Organize data by algorithm
        algo_data = defaultdict(lambda: {"budgets": [], "values": [], "errors": []})

        for budget in sorted(budgets_data.keys()):
            for algo, stats in budgets_data[budget].items():
                if stats[metric] is not None:
                    algo_data[algo]["budgets"].append(budget)
                    algo_data[algo]["values"].append(stats[metric])
                    algo_data[algo]["errors"].append(stats[se_key] if stats[se_key] is not None else 0)

        # Plot each algorithm
        for algo in sorted(algo_data.keys()):
            data = algo_data[algo]
            color = algorithm_colors.get(algo) if algorithm_colors else None

            ax.errorbar(
                data["budgets"],
                data["values"],
                yerr=np.array(data["errors"]) * 1.96,
                marker="o",
                label=algo,
                capsize=3,
                linewidth=1.5,
                markersize=1.6,
                color=color,
                alpha=0.8
            )

        ax.set_title(problem, fontsize=9)
        ax.set_xlabel("Budget", fontsize=8)
        if ax == axes[0]:
            ylabel = "Relative Error" if "relative" in metric else "Frobenius Distance"
            ax.set_ylabel(ylabel, fontsize=8)
        ax.tick_params(axis='both', labelsize=8)
        ax.grid(axis="y", linestyle=":", alpha=0.8, zorder=-1)
        sns.despine(ax=ax)

    handles, labels = axes[0].get_legend_handles_labels()

    left = .878
    top = 0.75

    if dataset == "modelnet10":
        top = 0.7
    if dataset == "tu_berlin":
        top = 0.665

    leg = fig.legend(
        handles, labels, title="",
        loc="right", ncol=len(labels) // 2,
        bbox_to_anchor=(left, top),
        fontsize=8, title_fontsize=8,
        columnspacing=0.3, framealpha=0.5, labelspacing=0.3,
    )

    #plt.tight_layout(pad=0.6, w_pad=0.3)

    plt.tight_layout()
    if figure_path is not None:
        os.makedirs(figure_path, exist_ok=True)
        pdf_path = os.path.join(figure_path, f"{save_name}.pdf")
        pgf_path = os.path.join(figure_path, f"{save_name}.pgf")
        plt.savefig(pdf_path, bbox_inches="tight")
        plt.savefig(pgf_path, bbox_inches="tight")
        print(f"Saved: {pdf_path}")

    plt.show()


# Plot relative error
plot_error_metrics_vs_budget(
    analysis=analysis,
    problem_names=problem_names_renamed,
    metric="mean_relative_error",
    se_key="se_relative_error",
    algorithm_colors=algorithm_colors,
    fig_width=W,
    figure_path=figure_path3,
    save_name="relative_error_vs_budget"
)

# Plot Frobenius distance
plot_error_metrics_vs_budget(
    analysis=analysis,
    problem_names=problem_names_renamed,
    metric="mean_frobenius",
    se_key="se_frobenius",
    algorithm_colors=algorithm_colors,
    fig_width=W,
    figure_path=figure_path3,
    save_name="frobenius_distance_vs_budget"
)

In [ ]:
#Wrapper to check evaluation counts used.

In [ ]:
class EvaluationCounterWrapper(torch.nn.Module):
    """
    Wrapper that counts forward and backward evaluations with batch sizes.
    """

    def __init__(self, model):
        super().__init__()
        self.model = model
        self.reset_counts()

    def reset_counts(self):
        self.forward_count = 0
        self.forward_samples = 0
        self.backward_count = 0
        self.backward_samples = 0

    def forward(self, x, *args, **kwargs):
        batch_size = x.shape[0] if isinstance(x, torch.Tensor) else x[0].shape[0]
        self.forward_count += 1
        self.forward_samples += batch_size

        output = self.model(x, *args, **kwargs)

        if output.requires_grad:
            output.register_hook(self._backward_hook(batch_size))

        return output

    def _backward_hook(self, batch_size):
        def hook(grad):
            self.backward_count += 1
            self.backward_samples += batch_size

        return hook

    def get_counts(self):
        return {
            "forward_passes": self.forward_count,
            "forward_samples": self.forward_samples,
            "backward_passes": self.backward_count,
            "backward_samples": self.backward_samples,
            "total_evaluations": self.forward_count + self.backward_count,
            "total_samples": self.forward_samples + self.backward_samples
        }

    def __getattr__(self, name):
        try:
            return super().__getattr__(name)
        except AttributeError:
            return getattr(self.model, name)


In [ ]:
# Create wrapped logit energy model
wrapped_model = EvaluationCounterWrapper(model)
logit_energy = SinglePassConfidence(wrapped_model, EnergyConfidence(), index=None)
problem_energy_logits_wrapped = TransformationProblem(
    logit_energy,
    transform_seq,
    consolidate_method="consolidate_simple"
)
problem_energy_logits_wrapped.max_batch_size = dataset_info.batch_size_search

# Get single sample from test set
x_single, y_single = next(iter(test_loader_transformed))
x_single = x_single[:1].to(device)
y_single = y_single[:1].to(device)

actual_costs = {}

model.eval().to(device)
for param in model.parameters():
    param.requires_grad = False

# Create proper DataLoader with (x, y) pairs
from torch.utils.data import TensorDataset, DataLoader as TorchDataLoader

single_dataset = TensorDataset(x_single, y_single)
single_loader = TorchDataLoader(single_dataset, batch_size=1, shuffle=False)



In [ ]:
import json
import os

# Define cache path
cache_dir = os.path.join(results_dir_path, "evaluation_costs_cache2")
os.makedirs(cache_dir, exist_ok=True)
cache_file = os.path.join(cache_dir, "evaluation_costs.json")

# Load cached costs if they exist
if os.path.exists(cache_file):
    with open(cache_file, 'r') as f:
        actual_costs = json.load(f)
    print(f"Loaded cached evaluation costs from {cache_file}")
else:
    actual_costs = {}

# Run each algorithm on single sample for each budget
for budget in budgets:
    for algo in all_algos:
        key = f"{algo}_budget_{budget}"

        # Skip if already cached
        if key in actual_costs and actual_costs[key] is not None and False:
            print(f"[{algo}] Budget {budget} - Using cached cost: {actual_costs[key]} samples")
            continue

        print(f"\n=== Algorithm: {algo}, Budget: {budget} ===")

        gc.collect()
        torch.cuda.empty_cache()

        # Map algorithm name
        algo_name_for_path = algo_variant_mapping.get(algo, algo)
        if algo == "shgo_individual":
            algo_name_for_path = "shgo"
        elif algo == "wcd_lat_ind":
            algo_name_for_path = "wcd_lattice"

        # Load best params from this budget
        algo_dir = os.path.join(base_results_dir, algo, f"budget_{budget}")
        param_path = os.path.join(algo_dir, "best.yml")

        stored_params = None
        if os.path.exists(param_path):
            stored_params = load_params(param_path)
        else:
            print(f"[{algo}] Budget {budget} - No stored params found, using defaults")
            default_params_kwargs = {}
            if algo in grad_weight_algos:
                default_params_kwargs["grad_weight"] = grad_weight
            stored_params = get_default_params(algo_name_for_path, budget, **default_params_kwargs)

        if algo == "random_search":
            stored_params = {}

        # Build search algorithm
        search_obj = build_search_algorithm(
            algo_name_for_path,
            stored_params,
            problem=problem_energy_logits_wrapped,
            budget=budget,
            model=model,
        )
        if algo == "its":
            search_obj.its.extend = 0

        # Reset counter before evaluation
        wrapped_model.reset_counts()

        # Run single evaluation
        try:
            metrics = load_or_run_evaluate_confidence_and_search(
                model=model,
                optimizer=search_obj,
                problem=problem_energy_logits_wrapped,
                test_loader=single_loader,
                save_path=None,
                max_batch_override=dataset_info.batch_size_search,
                show_progress=False,
                repeats=1,
                return_per_run=True,
                overwrite=True,
                store_val=False,
            )

            # Read evaluation count
            counts = wrapped_model.get_counts()
            actual_costs[key] = counts["total_samples"]
            print(f"[{algo}] Budget {budget} - Evaluation cost: {counts['total_samples']} samples")

        except Exception as e:
            print(f"[{algo}] Budget {budget} - Error: {e}")
            actual_costs[key] = None

        if not (dataset == "coil100" and algo == "its"):
            assert actual_costs[key] <= budget, f"Cost {actual_costs[key]} exceeds expected maximum for budget {budget}"
        # Save cache after each run
        with open(cache_file, 'w') as f:
            json.dump(actual_costs, f, indent=2)

        gc.collect()
        torch.cuda.empty_cache()

# Final display and save
print("\n=== Actual Evaluation Costs ===")
for key, cost in sorted(actual_costs.items()):
    if cost is not None:
        print(f"{key}: {cost} samples")
    else:
        print(f"{key}: Failed")

# Save final results
output_path = os.path.join(cache_dir, "evaluation_costs_final.json")
with open(output_path, 'w') as f:
    json.dump(actual_costs, f, indent=2)
print(f"\nEvaluation costs saved to: {output_path}")


